# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The dataset contains ordered logistic regression outputs for adoption predictors of indigenous and modern knowledge in rangeland management, covering socio-demographic characteristics and intervention outcomes.

### Dataset Source
Dataset schema is available at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`, referencing entities by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"Dataset Name: {metadata.name}")
print(f"Dataset Description: {metadata.description}")

## 2. Data Overview
List available record sets, their `@id`s, and fields (columns/attributes) referenced by their `@id`.

In [ ]:
# Obtain record sets via their @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        # Show fields for each record set by their @id
        fields = rs.get('field', [])
        for f in fields:
            print(f"  Field @id: {f['@id']}, name: {f.get('name', f['@id'])}")
        print()

## 3. Data Extraction
Load data from each record set into a DataFrame, referencing by their `@id`. Use the discovered record set and field `@id`s.

In [ ]:
# Collect recordSet @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet] if dataset.metadata.recordSet else []

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Example: Show columns of the first record set loaded
if dataframes:
    first_rs_id = next(iter(dataframes.keys()))
    print(f"Columns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No dataframes loaded. Check if record sets are properly defined.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalization, categorical grouping. Reference all fields and columns strictly by their `@id`s.

In [ ]:
# Select record set and numeric field by @id
# Replace the following @ids with actual values from the overview step above if they exist
example_record_set_id = None
example_numeric_field_id = None
example_group_field_id = None

# Assign from metadata (example only, you must update these to match real @ids)
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    # Try to pick a numeric field from the first record set
    fields = dataset.metadata.recordSet[0].get('field', [])
    numeric_fields = [f['@id'] for f in fields if f.get('dataType') in ('schema:Float', 'schema:Integer', 'schema:Number')]
    if numeric_fields:
        example_numeric_field_id = numeric_fields[0]
    # Try to pick a group field that is non-numeric
    group_fields = [f['@id'] for f in fields if f.get('dataType') not in ('schema:Float', 'schema:Integer', 'schema:Number')]
    if group_fields:
        example_group_field_id = group_fields[0]

if example_record_set_id and example_numeric_field_id:
    df = dataframes[example_record_set_id]
    # Filter records where numeric field > threshold
    threshold = df[example_numeric_field_id].mean() if not pd.isnull(df[example_numeric_field_id].mean()) else 10

    filtered_df = df[df[example_numeric_field_id] > threshold]
    print(f"Filtered records from {example_record_set_id} with {example_numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{example_numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
    print(f"Normalized {example_numeric_field_id} for filtered records:")
    print(filtered_df[[example_numeric_field_id, normalized_col]].head())

    # Group by selected group field if present
    if example_group_field_id and example_group_field_id in df.columns:
        grouped_df = filtered_df.groupby(example_group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {example_group_field_id}:")
        print(grouped_df.head())
else:
    print("Could not identify valid numeric and group fields for EDA. Please review the field @ids from above.")

## 5. Visualization
Visualize distributions and relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution if available
if example_record_set_id and example_numeric_field_id:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[example_numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {example_numeric_field_id} in {example_record_set_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Visualize relationship between numeric and group field if available
if example_record_set_id and example_numeric_field_id and example_group_field_id and example_group_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[example_group_field_id], y=df[example_numeric_field_id])
    plt.title(f"{example_numeric_field_id} by {example_group_field_id}")
    plt.xlabel(example_group_field_id)
    plt.ylabel(example_numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading the FAIR^2 Croissant schema dataset, extracting and summarizing record sets via their `@id`, and referencing all fields strictly by their `@id`s. The pipeline enables reproducible exploration, normalization, filtering, and visualization of logistic regression outputs.

Key findings:
- You can reference and handle all dataset entities using their unique `@id` for robust, portable data science workflows.
- The dataset is suitable for analysis of adoption predictors for rangeland management, but limitations exist regarding completeness and bias, as described in the metadata.
- Visualizations and grouping highlight the distribution of key numeric outcomes and categorical demographics.

Further improvements include deeper domain-specific feature engineering, bias analysis, and integration with downstream modeling.